In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/genshin_impact.csv")
df.head()

,Unnamed: 0,name,rarity,weapon,element,3D-model,region,release_date,days_since_release,banner_versions,...,lvl_90_ATK,lvl_90_DEF,duplicate_rate,c6_rate,avg_copies_per_player,num_banners,months_since_release,is_standard_banner,is_archon,pulled_count
0,0,lynette,4,Sword,Anemo,Medium Female,Fontaine,2023-08-16,872,"['4.0', '4.6', '5.6']",...,232,712,0.774796,0.036392,4.440421,3,28.590164,1,0,1338383
1,1,varesa,5,Catalyst,Electro,Medium Female,Natlan,2025-03-26,284,"['5.5', 'Luna III']",...,356,782,0.256102,0.036035,1.344270,2,9.311475,0,0,93049
2,2,chongyun,4,Claymore,Cryo,Medium Male,Liyue,2020-09-28,1924,"['1.1', '1.3', '2.0', '2.2', '2.4', '2.7', '3....",...,223,648,0.688872,0.023110,3.214113,10,63.081967,1,0,4277322
3,3,tighnari,5,Bow,Dendro,Medium Male,Sumeru,2022-08-24,1229,"['3.0', 'Luna I']",...,268,630,0.050633,0.001336,1.053333,2,40.295082,1,0,717953
4,4,jean,5,Sword,Anemo,Tall Female,Mondstadt,2020-09-28,1924,['4.5'],...,239,769,0.036889,0.000031,1.038302,1,63.081967,1,0,1341413


In [3]:
X = df.drop(columns=[
    "Unnamed: 0",
    "name",
    "rarity",
    "release_date",
    "banner_versions",
    "duplicate_rate",
    "c6_rate",
    "avg_copies_per_player",
    "pulled_count",
])

y = df["rarity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((82, 23), (21, 23), (82,), (21,))

In [4]:
from sklearn.compose import ColumnTransformer

num_cols = X.select_dtypes(
    include="number"
).columns.to_list()

cat_cols = X.select_dtypes(
    exclude="number"
).columns.to_list()

preprocessor = ColumnTransformer([
    ("numeric", num_pipe(), num_cols),
    ("categoric", cat_pipe(encoder="onehot"), cat_cols),
])

In [5]:
from xgboost import XGBClassifier

pipeline = Pipeline([
    ('prep', preprocessor),
    ('algo', XGBClassifier(
        n_jobs=-1,
        random_state=42,
        eval_metric='mlogloss'
    ))
])

In [6]:
from sklearn.model_selection import RandomizedSearchCV
from jcopml.tuning import random_search_params as rsp

model = RandomizedSearchCV(
    pipeline,
    rsp.xgb_params,
    cv=5,
    scoring='f1_weighted',
    n_iter=50,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

y_train = y_train.map({4: 0, 5: 1})
y_test = y_test.map({4: 0, 5: 1})

model.fit(X_train, y_train)

print(model.best_params_)
print(
    model.score(X_train, y_train),
    model.best_score_,
    model.score(X_test, y_test)
)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
{'algo__colsample_bytree': np.float64(0.12287721406968569), 'algo__gamma': 3, 'algo__learning_rate': np.float64(0.011557352816269864), 'algo__max_depth': 7, 'algo__n_estimators': 151, 'algo__reg_alpha': np.float64(0.179102763902448), 'algo__reg_lambda': np.float64(0.6054305695862118), 'algo__subsample': np.float64(0.36966572720293783)}
1.0 0.9875490196078431 0.9519327731092437


In [7]:
save_model(model, "ghensin.pkl")

Model is pickled as model/ghensin.pkl
